# Database Agent

In [13]:
#!pip install sqlalchemy mysql-connector-python langchain langchain-community langchain-ollama

In [14]:
from pprint import pprint
import sqlalchemy as db
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

## Set Up Agent

In [15]:
# Initialize the Ollama model (replace with the correct model name you have installed)
#llm = ChatOllama(model="llama3.2", base_url="http://localhost:11434")  

llm = ChatOllama(model="gemma4:e4b-mlx", base_url="http://localhost:11434")

In [16]:
# Create SQLAlchemy engine like in PythonMySQL
engine = db.create_engine("mysql+mysqlconnector://root:@localhost/classicmodels")

# Connect and verify connection
with engine.connect() as connection:
    print("Database connection successful.")

# Initialize LangChain SQLDatabase from SQLAlchemy engine
db = SQLDatabase(engine)

# Quick test
print("Dialect:", db.dialect)
print("Usable tables:", db.get_usable_table_names())
print(db.run("SELECT COUNT(*) FROM customers;"))

Database connection successful.
Dialect: mysql
Usable tables: ['customers', 'employees', 'offices', 'orderdetails', 'orders', 'payments', 'productlines', 'products']
[(122,)]


In [17]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()
for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [18]:
from langchain_core.prompts import PromptTemplate

# Standard SQL Agent system prompt (matches langchain-ai/sql-agent-system-prompt)
sql_agent_system_prompt = """You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the given tools. Only use the information returned by the tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.

If the question does not seem related to the database, just return "I don't know" as the answer.
"""

try:
    from langchain_classic import hub
    prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")
except Exception:
    prompt_template = PromptTemplate.from_template(sql_agent_system_prompt)

prompt_template

PromptTemplate(input_variables=['dialect', 'top_k'], input_types={}, partial_variables={}, template='You are an agent designed to interact with a SQL database.\nGiven an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.\nUnless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.\nYou can order the results by a relevant column to return the most interesting examples in the database.\nNever query for all the columns from a specific table, only ask for the relevant columns given the question.\nYou have access to tools for interacting with the database.\nOnly use the given tools. Only use the information returned by the tools to construct your final answer.\nYou MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\nDO NOT make any DML statements (INSERT, UPDATE, DE

In [19]:
#pprint(toolkit.get_context())

In [20]:
system_message = prompt_template.format(dialect=db.dialect, top_k=5)

In [21]:
# Add additional context - Table Information for all tables with sample rows
table_info = toolkit.get_context().get('table_info', '')
if table_info:
    system_message += f"\n\nTable Information:\n{table_info}"

In [22]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_message    
)

In [23]:
def executeStatement(query):
    events = agent.stream(
        {"messages": [("user", query)]},
        stream_mode="values",
    )
    for event in events:
        event["messages"][-1].pretty_print()

## Natural Language Queries

In [24]:
query = "What are the total number of employees ?"
executeStatement (query)

================================ Human Message =================================

What are the total number of employees ?
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (e0c639a3-ac13-4098-9d63-33e65dbd01fd)
 Call ID: e0c639a3-ac13-4098-9d63-33e65dbd01fd
  Args:
    query: SELECT COUNT(*) FROM employees;
================================= Tool Message =================================
Name: sql_db_query

[(23,)]
================================== Ai Message ==================================

There are a total of 23 employees.


In [25]:
query = "What are the total number customers ?"
executeStatement (query)

================================ Human Message =================================

What are the total number customers ?
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (db6d4d88-aa69-428c-9433-ff1434858aeb)
 Call ID: db6d4d88-aa69-428c-9433-ff1434858aeb
  Args:
    query: SELECT COUNT(customerNumber) FROM customers;
================================= Tool Message =================================
Name: sql_db_query

[(122,)]
================================== Ai Message ==================================

There are 122 customers.


In [26]:
query = "List a few customer details."
executeStatement (query)

================================ Human Message =================================

List a few customer details.
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (1c24e017-1e0e-4798-b23b-f436070c6c27)
 Call ID: 1c24e017-1e0e-4798-b23b-f436070c6c27
  Args:
    query: SELECT customerName, contactFirstName, city FROM customers LIMIT 5;
================================= Tool Message =================================
Name: sql_db_query

[('Atelier graphique', 'Carine ', 'Nantes'), ('Signal Gift Stores', 'Jean', 'Las Vegas'), ('Australian Collectors, Co.', 'Peter', 'Melbourne'), ('La Rochelle Gifts', 'Janine ', 'Nantes'), ('Baane Mini Imports', 'Jonas ', 'Stavern')]
================================== Ai Message ==================================

Here are a few customer details:

*   Atelier graphique, Carine , Nantes
*   Signal Gift Stores, Jean, Las Vegas
*   Australian Collectors, Co., Peter, Melbourne
*   La Rochelle Gifts, Janine 

In [27]:
query = "Give the total number of customers in each country."
executeStatement (query)

================================ Human Message =================================

Give the total number of customers in each country.
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (5e6ab174-6f60-4f8b-bfd3-89d0831ecd1d)
 Call ID: 5e6ab174-6f60-4f8b-bfd3-89d0831ecd1d
  Args:
    query: SELECT country, count(customerNumber) AS total_customers FROM customers GROUP BY country;
================================= Tool Message =================================
Name: sql_db_query

[('France', 12), ('USA', 36), ('Australia', 5), ('Norway', 3), ('Poland', 1), ('Germany', 13), ('Spain', 7), ('Sweden', 2), ('Denmark', 2), ('Singapore', 3), ('Portugal', 2), ('Japan', 2), ('Finland', 3), ('UK', 5), ('Ireland', 2), ('Canada', 3), ('Hong Kong', 1), ('Italy', 4), ('Switzerland', 3), ('Netherlands', 1), ('Belgium', 2), ('New Zealand', 4), ('South Africa', 1), ('Austria', 2), ('Philippines', 1), ('Russia', 1), ('Israel', 1)]
====================

In [28]:
query = "How many customers in Spain ?"
executeStatement (query)

================================ Human Message =================================

How many customers in Spain ?
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (ada47bd7-d555-4399-ac46-b497a0acd5ad)
 Call ID: ada47bd7-d555-4399-ac46-b497a0acd5ad
  Args:
    query: SELECT COUNT(customerNumber) FROM customers WHERE country = 'Spain'
================================= Tool Message =================================
Name: sql_db_query

[(7,)]
================================== Ai Message ==================================

There are 7 customers in Spain.


In [29]:
query = "List a few employees and their details."
executeStatement (query)

================================ Human Message =================================

List a few employees and their details.
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (d334032d-771d-49b8-ba25-26810edbbddd)
 Call ID: d334032d-771d-49b8-ba25-26810edbbddd
  Args:
    query: SELECT employeeNumber, firstName, lastName, jobTitle, email FROM employees LIMIT 5
================================= Tool Message =================================
Name: sql_db_query

[(1002, 'Diane', 'Murphy', 'President', 'dmurphy@classicmodelcars.com'), (1056, 'Mary', 'Patterson', 'VP Sales', 'mpatterso@classicmodelcars.com'), (1076, 'Jeff', 'Firrelli', 'VP Marketing', 'jfirrelli@classicmodelcars.com'), (1088, 'William', 'Patterson', 'Sales Manager (APAC)', 'wpatterson@classicmodelcars.com'), (1102, 'Gerard', 'Bondur', 'Sale Manager (EMEA)', 'gbondur@classicmodelcars.com')]
================================== Ai Message ==================================


In [30]:
query = "Who is the President?."
executeStatement (query)

================================ Human Message =================================

Who is the President?.
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (43bd897f-3dc1-4597-a7f4-df6ad8bfaac2)
 Call ID: 43bd897f-3dc1-4597-a7f4-df6ad8bfaac2
  Args:
    query: SELECT first_name, last_name FROM employees WHERE jobTitle = 'President' LIMIT 1
================================= Tool Message =================================
Name: sql_db_query

Error: (mysql.connector.errors.ProgrammingError) 1054 (42S22): Unknown column 'first_name' in 'field list'
[SQL: SELECT first_name, last_name FROM employees WHERE jobTitle = 'President' LIMIT 1]
(Background on this error at: https://sqlalche.me/e/20/f405)
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (45c39043-fd52-4561-b5a7-3d03f53b880c)
 Call ID: 45c39043-fd52-4561-b5a7-3d03f53b880c
  Args:
    table_names: employees
==============

In [31]:
query = "Who has the job title President?"
executeStatement (query)

================================ Human Message =================================

Who has the job title President?
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (4a21252a-2a31-46f1-bfa9-65845226940b)
 Call ID: 4a21252a-2a31-46f1-bfa9-65845226940b
  Args:
    query: SELECT firstName, lastName FROM employees WHERE jobTitle = 'President' LIMIT 1;
================================= Tool Message =================================
Name: sql_db_query

[('Diane', 'Murphy')]
================================== Ai Message ==================================

Diane Murphy has the job title President.


In [32]:
query = "What is the name of the customer that placed the most orders."
executeStatement (query)

================================ Human Message =================================

What is the name of the customer that placed the most orders.
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (e913467c-e5ff-42e8-8ee4-7a9bd3963e2a)
 Call ID: e913467c-e5ff-42e8-8ee4-7a9bd3963e2a
  Args:
    query: SELECT T1.customerName, COUNT(T2.orderNumber) AS order_count FROM customers AS T1 INNER JOIN orders AS T2 ON T1.customerNumber = T2.customerNumber GROUP BY T1.customerName ORDER BY order_count DESC LIMIT 1;
================================= Tool Message =================================
Name: sql_db_query

[('Euro+ Shopping Channel', 26)]
================================== Ai Message ==================================

The name of the customer that placed the most orders is Euro+ Shopping Channel.


In [33]:
query = "List of employees along with their managers and designation."
executeStatement (query)

================================ Human Message =================================

List of employees along with their managers and designation.
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (56a53825-e7cd-44a8-8145-8c17e05ad346)
 Call ID: 56a53825-e7cd-44a8-8145-8c17e05ad346
  Args:
    query: SELECT T1.firstName AS EmployeeFirstName, T1.lastName AS EmployeeLastName, T1.jobTitle, T2.firstName AS ManagerFirstName, T2.lastName AS ManagerLastName FROM employees AS T1 LEFT JOIN employees AS T2 ON T1.reportsTo = T2.employeeNumber;
================================= Tool Message =================================
Name: sql_db_query

[('Diane', 'Murphy', 'President', None, None), ('Mary', 'Patterson', 'VP Sales', 'Diane', 'Murphy'), ('Jeff', 'Firrelli', 'VP Marketing', 'Diane', 'Murphy'), ('William', 'Patterson', 'Sales Manager (APAC)', 'Mary', 'Patterson'), ('Gerard', 'Bondur', 'Sale Manager (EMEA)', 'Mary', 'Patterson'), ('Anthony',